In [2]:
import os

In [1]:
panel = '24-P10-A'
IMAGE_DIR = f"/Users/eagle/Documents/eagle-classification/normalized_images/{panel}/VI/"

In [5]:
files = os.listdir(IMAGE_DIR)
f = files[0]

In [7]:
f.split('.')[0].split('_')[-2]

'Cell019'

In [2]:
import pandas as pd


def sort_csv_by_filename(csv_file):
    df = pd.read_csv(csv_file)
    sort_col = "filename"
    df = df.sort_values(by=sort_col, kind="stable").reset_index(drop=True)
    df.to_csv(csv_file, index=False)
    print(f"Sorted '{csv_file}' by '{sort_col}' and overwrote the file.")

In [4]:
panel = '23-P09-C'
csv_file = f"OPENAI/{panel}/classification_results_VI_{panel}.csv"
sort_csv_by_filename(csv_file)
csv_file = f"OPENAI/{panel}/classification_results_VI_{panel}.csv"
sort_csv_by_filename(csv_file)

Sorted 'OPENAI/23-P09-C/classification_results_VI_23-P09-C.csv' by 'filename' and overwrote the file.
Sorted 'OPENAI/23-P09-C/classification_results_VI_23-P09-C.csv' by 'filename' and overwrote the file.


In [ ]:
# Compare VIT vs EL CSV content (filename + classification)

# Use existing dataframes if already loaded; otherwise read from csv paths
vit_cmp = vit_df.copy() if "vit_df" in globals() else pd.read_csv(vit_csv)
el_cmp = el_df.copy() if "el_df" in globals() else pd.read_csv(el_csv)

# Normalize filename so both formats can be compared
# (VIT appears to contain full/relative paths, EL contains basename only)
vit_cmp["filename_norm"] = vit_cmp["filename"].apply(os.path.basename)
el_cmp["filename_norm"] = el_cmp["filename"].apply(os.path.basename)

# Optional cleanup
vit_cmp["classification"] = vit_cmp["classification"].astype(str).str.strip()
el_cmp["classification"] = el_cmp["classification"].astype(str).str.strip()

# 1) Filename differences
vit_names = set(vit_cmp["filename_norm"])
el_names = set(el_cmp["filename_norm"])

only_in_vit = sorted(vit_names - el_names)
only_in_el = sorted(el_names - vit_names)

print(f"Rows in VIT: {len(vit_cmp)}, unique filenames: {len(vit_names)}")
print(f"Rows in EL : {len(el_cmp)}, unique filenames: {len(el_names)}")
print(f"Filenames only in VIT: {len(only_in_vit)}")
print(f"Filenames only in EL : {len(only_in_el)}")

# 2) Classification differences for matching filenames
merged_cmp = vit_cmp[["filename_norm", "classification"]].merge(
    el_cmp[["filename_norm", "classification"]],
    on="filename_norm",
    how="outer",
    suffixes=("_vit", "_el"),
)

cls_diff = merged_cmp[
    merged_cmp["classification_vit"].notna()
    & merged_cmp["classification_el"].notna()
    & (merged_cmp["classification_vit"] != merged_cmp["classification_el"])
].sort_values("filename_norm")

print(f"Classification mismatches on common filenames: {len(cls_diff)}")

# Overall same check (same filenames and same classifications)
are_same = (len(only_in_vit) == 0) and (len(only_in_el) == 0) and (len(cls_diff) == 0)
print(f"Are the two CSVs the same (by filename + classification)? {are_same}")

# Show samples of differences
if only_in_vit:
    print("\nSample filenames only in VIT:")
    print(only_in_vit[:10])

if only_in_el:
    print("\nSample filenames only in EL:")
    print(only_in_el[:10])

if not cls_diff.empty:
    print("\nSample classification differences:")
    print(cls_diff.head(10).to_string(index=False))

In [9]:
import os
# Compare VIT vs EL classification differences
vit_csv = f"OPENAI/{panel}/classification_results_VIT_{panel}.csv"
el_csv = f"OPENAI/{panel}/classification_results_EL_{panel}.csv"
# Use existing dataframes if present; otherwise read from CSV paths
vit_cmp = pd.read_csv(vit_csv)
el_cmp = pd.read_csv(el_csv)

# Normalize filenames (VIT may contain paths, EL may contain basename)
vit_cmp["filename"] = vit_cmp["filename"].apply(os.path.basename)
el_cmp["filename"] = el_cmp["filename"].apply(os.path.basename)
col = "classification"
# Clean classification text
vit_cmp[col] = vit_cmp[col].astype(str).str.strip()
el_cmp[col] = el_cmp[col].astype(str).str.strip()

# Merge on filename and find classification mismatches
merged_all = vit_cmp[["filename", col]].merge(
    el_cmp[["filename", col]],
    on="filename",
    how="outer",
    suffixes=("_vit", "_el"),
)

merged = merged_all[
    merged_all[f"{col}_vit"].notna()
    & merged_all[f"{col}_el"].notna()
    & (merged_all[f"{col}_vit"] != merged_all[f"{col}_el"])
].sort_values("filename").reset_index(drop=True)

print(f"Total VIT rows: {len(vit_cmp)}")
print(f"Total EL rows : {len(el_cmp)}")
print(f"Classification differences: {len(merged)}")

if merged.empty:
    print("No classification differences found.")
else:
    display(merged.head(20))

Total VIT rows: 540
Total EL rows : 540
Classification differences: 48


,filename,classification_vit,classification_el
0,23-P09-C1_EL_Cell001_normalized.tif,good,Dark
1,23-P09-C1_EL_Cell019_normalized.tif,crack,Dark
2,23-P09-C1_EL_Cell031_normalized.tif,good,Dark
3,23-P09-C1_EL_Cell033_normalized.tif,good,Dark
4,23-P09-C1_EL_Cell036_normalized.tif,good,Dark
5,23-P09-C1_EL_Cell057_normalized.tif,good,Dark
6,23-P09-C1_EL_Cell060_normalized.tif,good,Dark
7,23-P09-C1_EL_Cell069_normalized.tif,good,Dark
8,23-P09-C1_EL_Cell071_normalized.tif,good,Dark
9,23-P09-C1_EL_Cell101_normalized.tif,good,Dark


In [ ]:
compare two csv files for classification differences

In [6]:
compare two csv files for classification differences

KeyError: 'classification'